# Day 020 | Pandas — Data Cleaning & Missing Values

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `.isna()` / `.isnull()` | Detect missing values | Find boreholes with missing N-values |
| `.fillna()` | Fill missing values | Replace missing unit weight with default 18.0 kN/m³ |
| `.dropna()` | Remove rows with missing data | Drop incomplete soil layers |
| `.interpolate()` | Estimate missing values | Fill missing depths by linear interpolation |
| `.replace()` | Swap specific values | Replace erroneous N=999 (refusal) with NaN |
| `.duplicated()` | Find duplicate rows | Detect double-entered borehole readings |
| `.drop_duplicates()` | Remove duplicates | Keep only unique layer records |

---

## Our Geotechnical Problem

Real borehole data is **messy**. Drillers skip readings, lab technicians leave fields blank, data entry creates duplicates, and instruments occasionally report impossible values.

Before any ML model can learn from geotechnical data, we must **clean** it:

```
Raw borehole log:
  BH-01, 4.5m, N=___, Sand        ← missing N-value
  BH-02, 9.0m, N=-5,  Gravel      ← negative N? Instrument error!
  BH-01, 3.0m, N=8,   Silty Sand  ← duplicate entry
  BH-03, ___m, N=25,  Sand        ← missing depth
  BH-01, 12.0, N=999, Bedrock     ← 999 = refusal, not a real N-value
```

Today we will load a messy borehole CSV and transform it into **analysis-ready data**.

---

## Part 1: Loading Messy Data — First Look

Always start by **looking** at your data before cleaning. Key inspection tools:

| Method | What It Shows |
|--------|---------------|
| `.head()` | First 5 rows |
| `.info()` | Column types, non-null counts |
| `.describe()` | Statistics (reveals outliers) |
| `.shape` | (rows, columns) |

In [1]:
# ─────────────────────────────────────────────────────
# LOADING MESSY BOREHOLE DATA
# ─────────────────────────────────────────────────────

import pandas as pd
import numpy as np

# Load the messy CSV
df = pd.read_csv('messy_borehole_data.csv')

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nFirst 10 rows:")
df.head(10)

Shape: 30 rows x 8 columns

First 10 rows:


,borehole_id,depth_m,N_value,soil_type,water_content,unit_weight,fines_content,remarks
0,BH-01,1.5,5.0,Fill,12.3,16.2,35.0,Topsoil with roots
1,BH-01,3.0,8.0,Silty Sand,18.5,17.8,28.0,Grey-brown
2,BH-01,4.5,NaN,Sand,14.2,18.5,8.0,Medium dense
3,BH-01,6.0,18.0,Clayey Sand,22.1,NaN,45.0,Grey moist
4,BH-01,7.5,28.0,Sand,15.0,19.2,6.0,Dense brown
5,BH-01,9.0,35.0,Gravel,NaN,20.5,3.0,Very dense
6,BH-01,10.5,50.0,Weathered Rock,8.2,22.0,2.0,Hard grey
7,BH-01,12.0,999.0,Bedrock,5.1,25.0,1.0,Refusal
8,BH-02,1.5,3.0,Fill,14.8,15.8,NaN,Dark brown organic
9,BH-02,3.0,6.0,Clay,32.5,16.5,92.0,Soft grey


In [2]:
# ─────────────────────────────────────────────────────
# INSPECTING DATA TYPES AND MISSING VALUES
# ─────────────────────────────────────────────────────

# .info() reveals column types and how many non-null values
print("DataFrame Info:")
print("=" * 50)
df.info()
print()

# .describe() reveals statistics — look for impossible values
print("\nDescriptive Statistics:")
print("=" * 50)
df.describe()

DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   borehole_id    30 non-null     str    
 1   depth_m        29 non-null     float64
 2   N_value        27 non-null     float64
 3   soil_type      29 non-null     str    
 4   water_content  27 non-null     float64
 5   unit_weight    28 non-null     float64
 6   fines_content  28 non-null     float64
 7   remarks        28 non-null     str    
dtypes: float64(5), str(3)
memory usage: 2.6 KB


Descriptive Statistics:


,depth_m,N_value,water_content,unit_weight,fines_content
count,29.000000,27.000000,27.000000,28.000000,28.000000
mean,6.206897,58.037037,15.548148,19.271429,20.500000
std,3.465435,188.979136,7.534095,2.867682,25.546617
min,1.500000,-5.000000,4.200000,15.500000,1.000000
25%,3.000000,8.000000,11.000000,17.425000,3.750000
50%,6.000000,15.000000,14.800000,18.650000,6.500000
75%,9.000000,37.500000,18.500000,20.850000,32.000000
max,12.000000,999.000000,35.200000,25.500000,95.000000


**What problems can we spot?**

1. **Missing values** — Some columns have fewer non-null entries than total rows
2. **N_value min = -5** — Negative N-values are physically impossible
3. **N_value max = 999** — This is likely a refusal code, not a real blow count
4. **Possible duplicates** — We have 30 rows but only 4 boreholes

---

## Part 2: Detecting Missing Values — `.isna()` and `.isnull()`

Pandas represents missing values as `NaN` (Not a Number). `.isna()` returns `True` where data is missing.

| Method | Returns | Use Case |
|--------|---------|----------|
| `df.isna()` | Boolean DataFrame | See every missing cell |
| `df.isna().sum()` | Count per column | Quick summary of missingness |
| `df.isna().sum().sum()` | Total count | Overall data quality score |

In [3]:
# ─────────────────────────────────────────────────────
# DETECTING MISSING VALUES
# ─────────────────────────────────────────────────────

# Count missing values per column
missing = df.isna().sum()
missing_pct = (df.isna().sum() / len(df) * 100).round(1)

print("Missing Value Report:")
print("=" * 45)
for col in df.columns:
    count = missing[col]
    pct = missing_pct[col]
    bar = '█' * int(pct / 5) if pct > 0 else '✓'
    print(f"  {col:18s} {count:3d} missing ({pct:5.1f}%)  {bar}")

print(f"\nTotal missing cells: {df.isna().sum().sum()}")
print(f"Total cells:         {df.shape[0] * df.shape[1]}")
print(f"Completeness:        {(1 - df.isna().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.1f}%")

Missing Value Report:
  borehole_id          0 missing (  0.0%)  ✓
  depth_m              1 missing (  3.3%)  
  N_value              3 missing ( 10.0%)  ██
  soil_type            1 missing (  3.3%)  
  water_content        3 missing ( 10.0%)  ██
  unit_weight          2 missing (  6.7%)  █
  fines_content        2 missing (  6.7%)  █
  remarks              2 missing (  6.7%)  █

Total missing cells: 14
Total cells:         240
Completeness:        94.2%


In [4]:
# ─────────────────────────────────────────────────────
# WHICH ROWS HAVE MISSING DATA?
# ─────────────────────────────────────────────────────

# Find rows where N_value is missing — critical for SPT analysis
missing_n = df[df['N_value'].isna()]
print(f"Rows with missing N_value ({len(missing_n)} rows):")
print(missing_n[['borehole_id', 'depth_m', 'N_value', 'soil_type']])
print()

# Find rows where ANY value is missing
any_missing = df[df.isna().any(axis=1)]
print(f"Rows with ANY missing value ({len(any_missing)} rows):")
print(any_missing[['borehole_id', 'depth_m', 'N_value', 'soil_type']].to_string())

Rows with missing N_value (3 rows):
   borehole_id  depth_m  N_value    soil_type
2        BH-01      4.5      NaN         Sand
20       BH-03      6.0      NaN  Clayey Sand
26       BH-04      3.0      NaN         Clay

Rows with ANY missing value (14 rows):
   borehole_id  depth_m  N_value       soil_type
2        BH-01      4.5      NaN            Sand
3        BH-01      6.0     18.0     Clayey Sand
5        BH-01      9.0     35.0          Gravel
8        BH-02      1.5      3.0            Fill
11       BH-02      6.0     15.0            Sand
12       BH-02      7.5     22.0             NaN
14       BH-02     10.5     45.0  Weathered Rock
18       BH-03      3.0     12.0            Sand
20       BH-03      6.0      NaN     Clayey Sand
21       BH-03      NaN     25.0            Sand
23       BH-03     10.5     55.0  Weathered Rock
26       BH-04      3.0      NaN            Clay
27       BH-04      4.5      8.0      Silty Sand
29       BH-04      7.5     20.0            Sand


---

## Part 3: Handling Erroneous Values — `.replace()`

Before filling missing values, we need to fix **erroneous** values that look like real data but are not:

- **N=999** → Refusal code (drill hit bedrock), not a real blow count
- **N=-5** → Impossible negative value (instrument error or data entry mistake)

Strategy: Replace these with `NaN` first, then handle all missing values together.

In [5]:
# ─────────────────────────────────────────────────────
# REPLACING ERRONEOUS VALUES
# ─────────────────────────────────────────────────────

# Work on a copy so we can compare before/after
df_clean = df.copy()

# 1. Replace N=999 with NaN (refusal — not a real SPT value)
print("Before: N=999 rows")
print(df_clean[df_clean['N_value'] == 999][['borehole_id', 'depth_m', 'N_value', 'soil_type']])

df_clean['N_value'] = df_clean['N_value'].replace(999, np.nan)
print(f"\nReplaced N=999 with NaN")

# 2. Replace negative N-values with NaN (physically impossible)
neg_mask = df_clean['N_value'] < 0
print(f"\nNegative N-values found: {neg_mask.sum()}")
print(df_clean[neg_mask][['borehole_id', 'depth_m', 'N_value', 'soil_type']])

df_clean.loc[neg_mask, 'N_value'] = np.nan
print(f"\nReplaced negative N-values with NaN")

# Verify
print(f"\nN_value range after cleaning: {df_clean['N_value'].min():.0f} to {df_clean['N_value'].max():.0f}")
print(f"N_value missing count: {df_clean['N_value'].isna().sum()}")

Before: N=999 rows
  borehole_id  depth_m  N_value soil_type
7       BH-01     12.0    999.0   Bedrock

Replaced N=999 with NaN

Negative N-values found: 1
   borehole_id  depth_m  N_value soil_type
13       BH-02      9.0     -5.0    Gravel

Replaced negative N-values with NaN

N_value range after cleaning: 3 to 65
N_value missing count: 5


---

## Part 4: Removing Duplicates — `.duplicated()` and `.drop_duplicates()`

Duplicate rows happen when data is entered twice, or when merging overlapping datasets.

| Method | What It Does |
|--------|-------------|
| `df.duplicated()` | Boolean: True for duplicate rows |
| `df.duplicated(subset=[...])` | Check only specific columns |
| `df.drop_duplicates()` | Keep first occurrence, remove rest |
| `df.drop_duplicates(keep='last')` | Keep last occurrence |

In [6]:
# ─────────────────────────────────────────────────────
# DETECTING AND REMOVING DUPLICATES
# ─────────────────────────────────────────────────────

# Check for exact duplicate rows
dupes = df_clean.duplicated()
print(f"Exact duplicate rows: {dupes.sum()}")
print()

# Show the duplicates
print("Duplicate rows:")
print(df_clean[dupes][['borehole_id', 'depth_m', 'N_value', 'soil_type']])
print()

# Also check for duplicates based on borehole_id + depth_m
# (same borehole, same depth = likely a duplicate reading)
key_dupes = df_clean.duplicated(subset=['borehole_id', 'depth_m'], keep=False)
print(f"Rows with duplicate borehole + depth: {key_dupes.sum()}")
print(df_clean[key_dupes][['borehole_id', 'depth_m', 'N_value', 'soil_type']])
print()

# Remove exact duplicates (keep first occurrence)
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
rows_after = len(df_clean)
print(f"Removed {rows_before - rows_after} duplicate row(s)")
print(f"Shape: {df_clean.shape}")

Exact duplicate rows: 1

Duplicate rows:
   borehole_id  depth_m  N_value   soil_type
16       BH-01      3.0      8.0  Silty Sand

Rows with duplicate borehole + depth: 2
   borehole_id  depth_m  N_value   soil_type
1        BH-01      3.0      8.0  Silty Sand
16       BH-01      3.0      8.0  Silty Sand

Removed 1 duplicate row(s)
Shape: (29, 8)


---

## Part 5: Filling Missing Values — `.fillna()`

Different strategies for different columns:

| Strategy | When to Use | Geotechnical Example |
|----------|-------------|---------------------|
| Fill with constant | Known default value | Missing unit weight → 18.0 kN/m³ |
| Fill with mean | Numerical, roughly normal | Missing water content → column mean |
| Fill with median | Numerical, has outliers | Missing N-value → median for that soil type |
| Forward fill | Sequential/time data | Missing depth → carry forward from above |
| Interpolation | Continuous measurements | Missing depth → linear interpolation |

In [7]:
# ─────────────────────────────────────────────────────
# FILLING MISSING VALUES — Different strategies
# ─────────────────────────────────────────────────────

print("Missing values BEFORE filling:")
print(df_clean.isna().sum())
print()

# Strategy 1: Fill unit_weight with a typical default (18.0 kN/m³)
# This is a reasonable assumption for general soil
df_clean['unit_weight'] = df_clean['unit_weight'].fillna(18.0)
print("Filled missing unit_weight with 18.0 kN/m³ (typical soil)")

# Strategy 2: Fill water_content with the median of each soil_type
# This is smarter — clay has higher water content than sand
wc_medians = df_clean.groupby('soil_type')['water_content'].transform('median')
df_clean['water_content'] = df_clean['water_content'].fillna(wc_medians)
print("Filled missing water_content with median per soil type")

# Strategy 3: Fill fines_content with the median of each soil_type
fc_medians = df_clean.groupby('soil_type')['fines_content'].transform('median')
df_clean['fines_content'] = df_clean['fines_content'].fillna(fc_medians)
print("Filled missing fines_content with median per soil type")

# Strategy 4: Fill missing remarks with 'No remarks'
df_clean['remarks'] = df_clean['remarks'].fillna('No remarks')
print("Filled missing remarks with 'No remarks'")

print(f"\nMissing values AFTER filling:")
print(df_clean.isna().sum())

Missing values BEFORE filling:
borehole_id      0
depth_m          1
N_value          5
soil_type        1
water_content    3
unit_weight      2
fines_content    2
remarks          2
dtype: int64

Filled missing unit_weight with 18.0 kN/m³ (typical soil)
Filled missing water_content with median per soil type
Filled missing fines_content with median per soil type
Filled missing remarks with 'No remarks'

Missing values AFTER filling:
borehole_id      0
depth_m          1
N_value          5
soil_type        1
water_content    0
unit_weight      0
fines_content    0
remarks          0
dtype: int64


---

## Part 6: Interpolation — `.interpolate()`

For continuous measurements like depth, **interpolation** estimates missing values from neighboring data points. This is far better than using a mean or constant.

```
Depth:  1.5  3.0  4.5  NaN  7.5  9.0
                        ↑
              Linear interpolation: (4.5 + 7.5) / 2 = 6.0
```

In [8]:
# ─────────────────────────────────────────────────────
# INTERPOLATION — Estimating missing depths
# ─────────────────────────────────────────────────────

# Show the row with missing depth
print("Row with missing depth:")
print(df_clean[df_clean['depth_m'].isna()][['borehole_id', 'depth_m', 'N_value', 'soil_type']])
print()

# Interpolate depth within each borehole group
# (we don't want to interpolate across different boreholes!)
df_clean['depth_m'] = df_clean.groupby('borehole_id')['depth_m'].transform(
    lambda x: x.interpolate(method='linear')
)

# Also interpolate N_value within each borehole
df_clean['N_value'] = df_clean.groupby('borehole_id')['N_value'].transform(
    lambda x: x.interpolate(method='linear')
)

print("After interpolation:")
print(f"  Missing depth_m: {df_clean['depth_m'].isna().sum()}")
print(f"  Missing N_value: {df_clean['N_value'].isna().sum()}")
print()

# Show BH-03 to verify the interpolated depth
print("BH-03 after interpolation:")
print(df_clean[df_clean['borehole_id'] == 'BH-03'][['borehole_id', 'depth_m', 'N_value', 'soil_type']].to_string())

Row with missing depth:
   borehole_id  depth_m  N_value soil_type
21       BH-03      NaN     25.0      Sand

After interpolation:
  Missing depth_m: 0
  Missing N_value: 0

BH-03 after interpolation:
   borehole_id  depth_m  N_value       soil_type
17       BH-03      1.5      7.0            Fill
18       BH-03      3.0     12.0            Sand
19       BH-03      4.5     15.0            Sand
20       BH-03      6.0     20.0     Clayey Sand
21       BH-03      7.5     25.0            Sand
22       BH-03      9.0     40.0          Gravel
23       BH-03     10.5     55.0  Weathered Rock
24       BH-03     12.0     65.0         Bedrock


---

## Part 7: Dropping Incomplete Rows — `.dropna()`

Sometimes it is better to **remove** rows that are too incomplete rather than guessing values.

| Method | What It Does |
|--------|-------------|
| `df.dropna()` | Drop rows with ANY missing value |
| `df.dropna(subset=[...])` | Drop only if specified columns are missing |
| `df.dropna(thresh=N)` | Keep rows with at least N non-null values |
| `df.dropna(how='all')` | Drop only if ALL values are missing |

In [9]:
# ─────────────────────────────────────────────────────
# DROPPING ROWS — When filling doesn't make sense
# ─────────────────────────────────────────────────────

print("Remaining missing values:")
print(df_clean.isna().sum())
print(f"\nRows before dropna: {len(df_clean)}")

# Drop rows where critical columns (soil_type) are still missing
# N_value and depth_m are essential for SPT analysis
df_clean = df_clean.dropna(subset=['borehole_id', 'depth_m', 'N_value', 'soil_type'])

print(f"Rows after dropna:  {len(df_clean)}")
print(f"\nFinal missing values:")
print(df_clean.isna().sum())

Remaining missing values:
borehole_id      0
depth_m          0
N_value          0
soil_type        1
water_content    0
unit_weight      0
fines_content    0
remarks          0
dtype: int64

Rows before dropna: 29
Rows after dropna:  28

Final missing values:
borehole_id      0
depth_m          0
N_value          0
soil_type        0
water_content    0
unit_weight      0
fines_content    0
remarks          0
dtype: int64


---

## Part 8: Validating Cleaned Data

After cleaning, always **validate** the result. Check that:
1. No missing values remain in critical columns
2. Value ranges are physically reasonable
3. Expected number of rows/boreholes

In [10]:
# ─────────────────────────────────────────────────────
# VALIDATION — Is our cleaned data reasonable?
# ─────────────────────────────────────────────────────

print("DATA VALIDATION REPORT")
print("=" * 55)

# Check 1: No missing critical values
critical_cols = ['borehole_id', 'depth_m', 'N_value', 'soil_type']
missing_critical = df_clean[critical_cols].isna().sum().sum()
status = 'PASS' if missing_critical == 0 else 'FAIL'
print(f"  Missing critical values:     {status} ({missing_critical})")

# Check 2: N-values in reasonable range (0-100 for SPT)
n_min, n_max = df_clean['N_value'].min(), df_clean['N_value'].max()
status = 'PASS' if n_min >= 0 and n_max <= 100 else 'FAIL'
print(f"  N-value range [0-100]:       {status} ({n_min:.0f} to {n_max:.0f})")

# Check 3: Depths are positive
status = 'PASS' if df_clean['depth_m'].min() > 0 else 'FAIL'
print(f"  Depths positive:             {status} (min={df_clean['depth_m'].min():.1f})")

# Check 4: Unit weights in reasonable range (14-26 kN/m³)
uw_min, uw_max = df_clean['unit_weight'].min(), df_clean['unit_weight'].max()
status = 'PASS' if uw_min >= 14 and uw_max <= 26 else 'FAIL'
print(f"  Unit weight [14-26 kN/m3]:   {status} ({uw_min:.1f} to {uw_max:.1f})")

# Check 5: No duplicate entries
dupes = df_clean.duplicated(subset=['borehole_id', 'depth_m']).sum()
status = 'PASS' if dupes == 0 else 'FAIL'
print(f"  No duplicate entries:        {status} ({dupes} dupes)")

# Check 6: Expected boreholes present
bh_count = df_clean['borehole_id'].nunique()
print(f"  Unique boreholes:            {bh_count}")

print("=" * 55)
print(f"  Final shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")

DATA VALIDATION REPORT
  Missing critical values:     PASS (0)
  N-value range [0-100]:       PASS (3 to 65)
  Depths positive:             PASS (min=1.5)
  Unit weight [14-26 kN/m3]:   PASS (15.5 to 25.5)
  No duplicate entries:        PASS (0 dupes)
  Unique boreholes:            4
  Final shape: 28 rows x 8 columns


---

## Part 9: Complete Application — Clean Borehole Data Pipeline

Let us combine everything into a **reusable cleaning pipeline** that takes raw borehole data and produces analysis-ready output.

In [11]:
# ══════════════════════════════════════════════════════
# COMPLETE: BOREHOLE DATA CLEANING PIPELINE
# ══════════════════════════════════════════════════════

def clean_borehole_data(filepath):
    """Clean messy borehole data and return analysis-ready DataFrame.
    
    Steps:
    1. Load data
    2. Replace erroneous values (N=999, negative N)
    3. Remove exact duplicates
    4. Fill missing values (group-aware strategies)
    5. Interpolate depths and N-values within each borehole
    6. Drop rows still missing critical fields
    7. Reset index
    """
    # Step 1: Load
    raw = pd.read_csv(filepath)
    print(f"Loaded: {raw.shape[0]} rows x {raw.shape[1]} columns")
    
    df = raw.copy()
    
    # Step 2: Replace erroneous values
    df['N_value'] = df['N_value'].replace(999, np.nan)
    df.loc[df['N_value'] < 0, 'N_value'] = np.nan
    print(f"  Replaced erroneous N-values with NaN")
    
    # Step 3: Remove duplicates
    n_dupes = df.duplicated().sum()
    df = df.drop_duplicates()
    print(f"  Removed {n_dupes} duplicate rows")
    
    # Step 4: Fill missing values
    df['unit_weight'] = df['unit_weight'].fillna(18.0)
    df['remarks'] = df['remarks'].fillna('No remarks')
    
    # Group-aware fills
    for col in ['water_content', 'fines_content']:
        medians = df.groupby('soil_type')[col].transform('median')
        df[col] = df[col].fillna(medians)
    print(f"  Filled missing values with group medians")
    
    # Step 5: Interpolate within each borehole
    for col in ['depth_m', 'N_value']:
        df[col] = df.groupby('borehole_id')[col].transform(
            lambda x: x.interpolate(method='linear')
        )
    print(f"  Interpolated depth and N-value within boreholes")
    
    # Step 6: Drop still-incomplete rows
    before = len(df)
    df = df.dropna(subset=['borehole_id', 'depth_m', 'N_value', 'soil_type'])
    print(f"  Dropped {before - len(df)} rows still missing critical data")
    
    # Step 7: Reset index
    df = df.reset_index(drop=True)
    
    print(f"\nResult: {df.shape[0]} rows x {df.shape[1]} columns (analysis-ready)")
    return raw, df


# Run the pipeline
print("BOREHOLE DATA CLEANING PIPELINE")
print("=" * 55)
raw_df, clean_df = clean_borehole_data('messy_borehole_data.csv')

BOREHOLE DATA CLEANING PIPELINE
Loaded: 30 rows x 8 columns
  Replaced erroneous N-values with NaN
  Removed 1 duplicate rows
  Filled missing values with group medians
  Interpolated depth and N-value within boreholes
  Dropped 1 rows still missing critical data

Result: 28 rows x 8 columns (analysis-ready)


In [12]:
# ─────────────────────────────────────────────────────
# BEFORE vs AFTER COMPARISON
# ─────────────────────────────────────────────────────

print("BEFORE vs AFTER")
print("=" * 55)
print(f"  {'Metric':<30s} {'Raw':>10s} {'Clean':>10s}")
print(f"  {'─' * 52}")
print(f"  {'Rows':<30s} {raw_df.shape[0]:>10d} {clean_df.shape[0]:>10d}")
print(f"  {'Missing cells':<30s} {raw_df.isna().sum().sum():>10d} {clean_df.isna().sum().sum():>10d}")
print(f"  {'Duplicate rows':<30s} {raw_df.duplicated().sum():>10d} {clean_df.duplicated().sum():>10d}")
print(f"  {'N-value min':<30s} {raw_df['N_value'].min():>10.0f} {clean_df['N_value'].min():>10.1f}")
print(f"  {'N-value max':<30s} {raw_df['N_value'].max():>10.0f} {clean_df['N_value'].max():>10.1f}")
print(f"  {'Unique boreholes':<30s} {raw_df['borehole_id'].nunique():>10d} {clean_df['borehole_id'].nunique():>10d}")
print("=" * 55)
print()

# Show the cleaned data
print("Cleaned data (first 12 rows):")
clean_df.head(12)

BEFORE vs AFTER
  Metric                                Raw      Clean
  ────────────────────────────────────────────────────
  Rows                                   30         28
  Missing cells                          14          0
  Duplicate rows                          1          0
  N-value min                            -5        3.0
  N-value max                           999       65.0
  Unique boreholes                        4          4

Cleaned data (first 12 rows):


,borehole_id,depth_m,N_value,soil_type,water_content,unit_weight,fines_content,remarks
0,BH-01,1.5,5.0,Fill,12.30,16.2,35.0,Topsoil with roots
1,BH-01,3.0,8.0,Silty Sand,18.50,17.8,28.0,Grey-brown
2,BH-01,4.5,13.0,Sand,14.20,18.5,8.0,Medium dense
3,BH-01,6.0,18.0,Clayey Sand,22.10,18.0,45.0,Grey moist
4,BH-01,7.5,28.0,Sand,15.00,19.2,6.0,Dense brown
5,BH-01,9.0,35.0,Gravel,9.85,20.5,3.0,Very dense
6,BH-01,10.5,50.0,Weathered Rock,8.20,22.0,2.0,Hard grey
7,BH-01,12.0,50.0,Bedrock,5.10,25.0,1.0,Refusal
8,BH-02,1.5,3.0,Fill,14.80,15.8,35.0,Dark brown organic
9,BH-02,3.0,6.0,Clay,32.50,16.5,92.0,Soft grey


In [13]:
# ─────────────────────────────────────────────────────
# QUICK ANALYSIS OF CLEANED DATA
# ─────────────────────────────────────────────────────

# Now we can confidently analyze the clean data
print("SITE SUMMARY FROM CLEANED DATA")
print("=" * 55)

for bh_id in clean_df['borehole_id'].unique():
    bh = clean_df[clean_df['borehole_id'] == bh_id]
    print(f"\n  {bh_id}:")
    print(f"    Layers:    {len(bh)}")
    print(f"    Depth:     {bh['depth_m'].min():.1f} to {bh['depth_m'].max():.1f} m")
    print(f"    N-value:   {bh['N_value'].min():.0f} to {bh['N_value'].max():.0f} (avg={bh['N_value'].mean():.1f})")
    print(f"    Soils:     {', '.join(bh['soil_type'].unique())}")

print(f"\n{'=' * 55}")
print(f"Total: {clean_df['borehole_id'].nunique()} boreholes, {len(clean_df)} layers")
print(f"Data is now CLEAN and ready for ML analysis!")

SITE SUMMARY FROM CLEANED DATA

  BH-01:
    Layers:    8
    Depth:     1.5 to 12.0 m
    N-value:   5 to 50 (avg=25.9)
    Soils:     Fill, Silty Sand, Sand, Clayey Sand, Gravel, Weathered Rock, Bedrock

  BH-02:
    Layers:    7
    Depth:     1.5 to 12.0 m
    N-value:   3 to 55 (avg=23.9)
    Soils:     Fill, Clay, Silty Sand, Sand, Gravel, Weathered Rock, Bedrock

  BH-03:
    Layers:    8
    Depth:     1.5 to 12.0 m
    N-value:   7 to 65 (avg=29.9)
    Soils:     Fill, Sand, Clayey Sand, Gravel, Weathered Rock, Bedrock

  BH-04:
    Layers:    5
    Depth:     1.5 to 7.5 m
    N-value:   4 to 20 (avg=10.4)
    Soils:     Fill, Clay, Silty Sand, Sand

Total: 4 boreholes, 28 layers
Data is now CLEAN and ready for ML analysis!


---

## Practice Exercises

### Exercise 1: Missing value detection
Load the messy CSV again. Find which borehole has the **most** missing values across all columns. Print its ID and the count.

### Exercise 2: Custom fill strategy
Instead of filling missing `unit_weight` with a constant 18.0, fill it with the **mean unit weight per soil type**. Compare the results.

### Exercise 3: Outlier replacement
Define outliers as N-values more than 2 standard deviations from the mean. Replace them with NaN, then interpolate.

In [14]:
# EXERCISE 1
# df_raw = pd.read_csv('messy_borehole_data.csv')
# missing_per_bh = df_raw.groupby('borehole_id').apply(lambda g: g.isna().sum().sum())
# print(missing_per_bh)
# print(f"Most missing: {missing_per_bh.idxmax()} with {missing_per_bh.max()} missing values")

In [15]:
# EXERCISE 2
# uw_means = df_raw.groupby('soil_type')['unit_weight'].transform('mean')
# df_raw['unit_weight_filled'] = df_raw['unit_weight'].fillna(uw_means)
# print(df_raw[['soil_type', 'unit_weight', 'unit_weight_filled']])

In [16]:
# EXERCISE 3
# mean_n = df_raw['N_value'].mean()
# std_n = df_raw['N_value'].std()
# outliers = (df_raw['N_value'] - mean_n).abs() > 2 * std_n
# print(f"Outliers found: {outliers.sum()}")
# df_raw.loc[outliers, 'N_value'] = np.nan
# df_raw['N_value'] = df_raw['N_value'].interpolate()

---

## Key Takeaways

1. **Always inspect data first** with `.info()`, `.describe()`, and `.isna().sum()` before doing anything else. Borehole data almost always has missing values and errors.

2. **Replace erroneous values** (N=999 refusal codes, negative values) with NaN using `.replace()` and boolean masking before attempting to fill missing data.

3. **Group-aware filling** (`.groupby().transform('median')`) is smarter than column-wide fills — clay water content differs from sand water content.

4. **Interpolation** (`.interpolate()`) is ideal for continuous geotechnical measurements like depth and N-value, especially when applied within each borehole group.

5. **Data cleaning is the foundation** of every ML pipeline. Garbage in = garbage out. A clean dataset is worth more than a fancy model.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| `.isna()` detection | Feature completeness check before training |
| `.fillna(median)` | Imputation — `sklearn.impute.SimpleImputer(strategy='median')` |
| `.interpolate()` | `sklearn.impute.KNNImputer` for smarter gap-filling |
| `.replace()` erroneous values | Outlier treatment in preprocessing |
| `.drop_duplicates()` | Preventing data leakage from repeated samples |
| `.dropna()` | Handling NaN before model training (most models cannot handle NaN) |
| Group-aware filling | Domain-informed imputation — better than blind statistics |
| Validation checks | Data quality assertions in ML pipelines |

Every ML model requires clean input data. The techniques from today — detection, replacement, imputation, interpolation, and validation — form the first stage of **every** geotechnical ML pipeline.

---

*Day 020 of 100 — Clean data is the foundation of every good model!*